<a href="https://colab.research.google.com/github/Will1202/Internship-learning-note/blob/Advanced-OCR-Comparison-and-Layout-Aware-Extraction/Layout_OCR_Demo_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PaddleOCR Quick Guide for Google Colab
# =====================================
# This notebook provides a complete guide for using PaddleOCR to extract text from PDFs and images

# ## 📋 Table of Contents
# 1. [Installation](#installation)
# 2. [Basic Setup](#setup)
# 3. [PDF to Image Conversion](#pdf-conversion)
# 4. [OCR Processing](#ocr-processing)
# 5. [Results Visualization](#visualization)
# 6. [Advanced Features](#advanced)

---



## 🚀 1. Installation {#installation}

First, let's install all required packages

In [ ]:
# Install PaddleOCR and dependencies
!pip install paddleocr
!pip install paddlepaddle



print("✅ All packages installed successfully!")

In [ ]:
# Install PDF processing tools
!pip install pdf2image
!apt-get install poppler-utils -y  # Required for pdf2image

## 🔧 2. Basic Setup
Import libraries and set up the environment

In [ ]:
# Import required libraries
from paddleocr import PaddleOCR
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2
import numpy as np
from pdf2image import convert_from_path
import os
from google.colab import files
import io

In [ ]:
# Upload your PDF file
print("📁 Upload your PDF file:")
uploaded = files.upload()

In [ ]:
# Get the uploaded file name
pdf_filename = list(uploaded.keys())[0]
pdf_path = f"/content/{pdf_filename}"
print(f"✅ File uploaded: {pdf_filename}")

---
## 🖼️ 3. PDF to Image Conversion {#pdf-conversion}
Convert PDF pages to images for OCR processing

In [ ]:
# Convert PDF to images
def convert_pdf_to_images(pdf_path, dpi=300):
    """
    Convert PDF pages to images

    Args:
        pdf_path (str): Path to PDF file
        dpi (int): Resolution for conversion (higher = better quality, larger file)

    Returns:
        list: List of PIL Image objects
    """
    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        print(f"✅ Successfully converted {len(images)} page(s)")
        return images
    except Exception as e:
        print(f"❌ Error converting PDF: {e}")
        return []



In [ ]:
# Convert the uploaded PDF
images = convert_pdf_to_images(pdf_path)

In [ ]:
# Display converted pages
def display_pdf_pages(images, max_pages=3):
    """Display the first few pages of the converted PDF"""
    pages_to_show = min(len(images), max_pages)

    fig, axes = plt.subplots(1, pages_to_show, figsize=(5*pages_to_show, 7))
    if pages_to_show == 1:
        axes = [axes]

    for i in range(pages_to_show):
        axes[i].imshow(images[i])
        axes[i].set_title(f'Page {i+1}')
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()



In [ ]:
if images:
    display_pdf_pages(images)
else:
    print("❌ No images to display")

---
## 🔍 4. OCR Processing {#ocr-processing}
Initialize PaddleOCR and process the images

In [ ]:
# Initialize PaddleOCR
def initialize_ocr(language='en', use_gpu=False):
    """
    Initialize PaddleOCR with specified settings

    Args:
        language (str): Language code ('en', 'ch', 'fr', etc.)
        use_gpu (bool): Whether to use GPU acceleration

    Returns:
        PaddleOCR: Initialized OCR object
    """
    try:
        ocr = PaddleOCR(
            use_textline_orientation=True,  # Enable text angle classification
            lang=language,       # Set language
        )
        print(f"✅ PaddleOCR initialized for language: {language}")
        return ocr
    except Exception as e:
        print(f"❌ Error initializing OCR: {e}")
        return None

In [ ]:
# Initialize OCR
ocr = initialize_ocr('en')

In [ ]:
# Process images with OCR
def process_image_ocr(ocr, image, page_num=1):
    """
    Process a single image with OCR

    Args:
        ocr: PaddleOCR object
        image: PIL Image object
        page_num (int): Page number for identification

    Returns:
        tuple: (results, processed_image_path)
    """
    # Save image temporarily
    img_path = f'/content/page_{page_num}.png'
    image.save(img_path, 'PNG')

    # Perform OCR
    try:
        result = ocr.ocr(img_path)
        print(f"✅ OCR completed for page {page_num}")
        return result, img_path
    except Exception as e:
        print(f"❌ OCR error for page {page_num}: {e}")
        return None, img_path

In [ ]:
# Process all pages
all_results = []
all_image_paths = []

for i, image in enumerate(images):
    result, img_path = process_image_ocr(ocr, image, i+1)
    all_results.append(result)
    all_image_paths.append(img_path)

---
## 📊 5. Results Visualization {#visualization}
Visualize OCR results with bounding boxes and extracted text

In [ ]:
# Visualization functions
def parse_ocr_result(ocr_result):
    """
    Parse OCR result and extract boxes, texts, and scores safely
    Handle both old format and new dictionary format

    Args:
        ocr_result: OCR results from PaddleOCR

    Returns:
        tuple: (boxes, texts, scores) or (None, None, None) if no valid results
    """
    if not ocr_result:
        return None, None, None


    # Handle list containing dictionary (newer PaddleOCR format)
    if isinstance(ocr_result, list) and len(ocr_result) > 0:
        first_element = ocr_result[0]

        # If first element is a dictionary, extract from it
        if isinstance(first_element, dict):

            if 'rec_texts' in first_element and 'rec_scores' in first_element and 'rec_polys' in first_element:
                boxes = first_element['rec_polys']
                txts = first_element['rec_texts']
                scores = first_element['rec_scores']

                return boxes, txts, scores
            else:
                return None, None, None

        # Handle traditional list format [[box, [text, score]], ...]
        elif isinstance(first_element, list):

            boxes = []
            txts = []
            scores = []

            for i, line in enumerate(ocr_result[0]):
                try:
                    if len(line) >= 2:
                        box = line[0]  # Bounding box coordinates

                        # Check if line[1] is a tuple/list with text and score
                        if isinstance(line[1], (list, tuple)) and len(line[1]) >= 2:
                            txt = line[1][0]  # Text
                            score = line[1][1]  # Confidence score
                        elif isinstance(line[1], str):
                            txt = line[1]
                            score = 1.0  # Default score
                        else:
                            continue

                        boxes.append(box)
                        txts.append(txt)
                        scores.append(score)

                except Exception as e:
                    print(f"Error parsing line {i}: {e}")
                    continue

            return boxes, txts, scores

    # Handle direct dictionary format
    elif isinstance(ocr_result, dict):

        if 'rec_texts' in ocr_result and 'rec_scores' in ocr_result and 'rec_polys' in ocr_result:
            boxes = ocr_result['rec_polys']
            txts = ocr_result['rec_texts']
            scores = ocr_result['rec_scores']

            return boxes, txts, scores
        else:

            return None, None, None

    print("Debug - Unrecognized OCR result format")
    return None, None, None

def draw_ocr_results(image_path, ocr_result):
    """
    Draw bounding boxes and text on the image

    Args:
        image_path (str): Path to the image file
        ocr_result: OCR results from PaddleOCR

    Returns:
        tuple: (annotated_image, boxes, txts, scores) or None if no results
    """
    boxes, txts, scores = parse_ocr_result(ocr_result)

    if not boxes:
        print("No valid OCR results to draw")
        return None

    # Load image
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)


    # Draw bounding boxes and text
    for i, (box, txt, score) in enumerate(zip(boxes, txts, scores)):
        if score > 0.5:  # Lowered threshold to see more results
            try:


                # Handle different box formats
                if isinstance(box, np.ndarray):
                    # Convert numpy array to list for easier handling
                    box_points = box.tolist()
                else:
                    box_points = box

                # Ensure we have 4 corner points
                if len(box_points) >= 4:
                    # PaddleOCR typically returns 4 corner points: [top-left, top-right, bottom-right, bottom-left]
                    # Each point is [x, y]
                    if isinstance(box_points[0], (list, tuple, np.ndarray)) and len(box_points[0]) == 2:
                        # Format: [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
                        corners = [(int(point[0]), int(point[1])) for point in box_points[:4]]
                    else:
                        # Flatten format: [x1,y1,x2,y2,x3,y3,x4,y4]
                        flat_coords = box_points[:8]  # Take first 8 coordinates
                        corners = [(int(flat_coords[j]), int(flat_coords[j+1])) for j in range(0, 8, 2)]


                    # Draw the polygon (quadrilateral bounding box)
                    if len(corners) >= 4:
                        # Draw lines between consecutive corners and close the shape
                        for j in range(len(corners)):
                            start_point = corners[j]
                            end_point = corners[(j + 1) % len(corners)]  # Wrap around to first point
                            draw.line([start_point, end_point], fill='red', width=2)

                        # Add confidence score at top-left corner
                        try:
                            font = ImageFont.load_default()
                            text_x, text_y = corners[0]  # Use first corner (typically top-left)
                            draw.text((text_x, text_y - 15), f"{score:.2f}", fill='blue', font=font)
                        except Exception as font_error:
                            print(f"Font error: {font_error}")
                            pass
                    else:
                        print(f"Warning - Not enough corners for box {i}: {len(corners)}")
                else:
                    print(f"Warning - Box {i} doesn't have enough points: {len(box_points)}")

            except Exception as e:
                print(f"Error drawing box {i} for text '{txt[:20]}': {e}")
                print(f"Box data: {box}")
                continue

    return image, boxes, txts, scores

def display_ocr_results(image_path, ocr_result, page_num):
    """Display original image and OCR results side by side"""
    print(f"Starting visualization for page {page_num}")

    result_data = draw_ocr_results(image_path, ocr_result)

    if result_data is None:
        print(f"❌ No text detected on page {page_num}")
        return None, None, None

    annotated_image, boxes, txts, scores = result_data
    original_image = Image.open(image_path)

    # Display images
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    ax1.imshow(original_image)
    ax1.set_title(f'Original - Page {page_num}')
    ax1.axis('off')

    ax2.imshow(annotated_image)
    ax2.set_title(f'OCR Results - Page {page_num} ({len(txts)} items found)')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

    return boxes, txts, scores



In [ ]:
# Process and display results for each page
extracted_text_all = []

for i, (result, img_path) in enumerate(zip(all_results, all_image_paths)):
    page_num = i + 1

    # Check if we have any OCR results
    if result:
        try:
            # Display visualization
            visualization_result = display_ocr_results(img_path, result, page_num)

            if visualization_result[0] is not None:  # Check if visualization succeeded
                boxes, txts, scores = visualization_result

                # Extract and display text
                page_text = []


                for j, (txt, score) in enumerate(zip(txts, scores)):
                    if score > 0.3:  # Lowered threshold to see more results
                        page_text.append(txt)

                extracted_text_all.extend(page_text)

            else:
                print(f"❌ Failed to process OCR results for page {page_num}")

        except Exception as e:
            print(f"❌ Error processing page {page_num}: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"❌ No OCR results for page {page_num}")

print(f"\n🔍 Total text segments extracted: {len(extracted_text_all)}")

# Display all extracted text
if extracted_text_all:
    print(f"\n📋 All Extracted Text:")
    print("=" * 50)
    for i, text in enumerate(extracted_text_all, 1):
        print(f"{i:3d}. {text}")
else:
    print("❌ No text was extracted from any page")

---
# 🌟 Alternative: EasyOCR (Quick & Modern)

EasyOCR is a newer, more specialized OCR engine optimized for documents

## Install EasyOCR

In [ ]:
# Install specific compatible versions
# This combination is known to work well with SuryaOCR and transformers
!pip install easyocr

In [ ]:
# Install
!pip install pdf2image
!apt-get install poppler-utils -y

## Import and setup SuryaOCR

In [ ]:
# Upload PDF and convert to image
from google.colab import files
from pdf2image import convert_from_path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import easyocr


In [ ]:
# Upload PDF
print("📁 Upload your PDF:")
uploaded = files.upload()
pdf_path = f"/content/{list(uploaded.keys())[0]}"

# Convert to image
images = convert_from_path(pdf_path, dpi=300)
image = images[0]  # Use first page
# Save image as file (EasyOCR works better with file paths)
image_path = '/content/page.png'
image.save(image_path)

# Show uploaded image
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title('Uploaded PDF - First Page')
plt.axis('off')
plt.show()

In [ ]:
# EasyOCR Processing
print("🔍 Running EasyOCR...")

# Create reader (English)
reader = easyocr.Reader(['en'])

# Extract text
result = reader.readtext(image_path)

In [ ]:
# Draw bounding boxes and extract text
img_copy = image.copy()
draw = ImageDraw.Draw(img_copy)
extracted_text = []


for (bbox, text, confidence) in result:
    if confidence > 0.5:  # Filter by confidence
        # Draw bounding box
        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))
        draw.rectangle([top_left, bottom_right], outline='red', width=2)

        # Add confidence score
        draw.text((top_left[0], top_left[1]-20), f"{confidence:.2f}", fill='red')

        extracted_text.append(text)

In [ ]:

# Display results
plt.figure(figsize=(15, 10))
plt.imshow(img_copy)
plt.title(f'EasyOCR Results - {len(extracted_text)} text segments found')
plt.axis('off')
plt.show()

In [ ]:
# Print extracted text
print(f"\n📝 Extracted Text ({len(extracted_text)} segments):")
print("-" * 50)
for i, text in enumerate(extracted_text, 1):
    print(f"{i:2d}. {text}")

print(f"\n✅ EasyOCR completed successfully!")

In [ ]:
!pip install pytesseract pdf2image
!apt-get install tesseract-ocr -y
!apt-get install poppler-utils -y
from google.colab import files
from pdf2image import convert_from_path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import pytesseract

# Upload PDF
print("📁 Upload your PDF:")
uploaded = files.upload()
pdf_path = f"/content/{list(uploaded.keys())[0]}"

# Convert to image
images = convert_from_path(pdf_path, dpi=300)
image = images[0]  # Use first page
image_path = '/content/page.png'
image.save(image_path)

# Show uploaded image
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title('Uploaded PDF - First Page')
plt.axis('off')
plt.show()

# Tesseract Processing
print("🔍 Running Tesseract...")

# Configure Tesseract to output bounding boxes
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'
custom_config = r'--oem 3 --psm 6 -l eng outputbase hocr'

# Extract text with bounding boxes
hocr_output = pytesseract.image_to_pdf_or_hocr(image_path, extension='hocr', config=custom_config)

# Parse HOCR output to get bounding boxes and text
from bs4 import BeautifulSoup

soup = BeautifulSoup(hocr_output, 'html.parser')
ocr_results = soup.find_all('span', class_='ocrx_word')

# Create image copy for drawing
img_copy = image.copy()
draw = ImageDraw.Draw(img_copy)
extracted_text = []

# Process each recognized word
for word in ocr_results:
    # Get bounding box coordinates
    title = word.get('title')
    if title:
        # Extract coordinates from title attribute
        coords = title.split(';')[0].split(' ')[1:]
        left, top,right,bottom  = map(int, coords)
        # Draw bounding box
        draw.rectangle([left, top, right, bottom], outline='red', width=2)
        extracted_text.append(word.text.strip())

# Display results
plt.figure(figsize=(15, 10))
plt.imshow(img_copy)
plt.title(f'Tesseract Results - {len(extracted_text)} text segments found')
plt.axis('off')
plt.show()

# Print extracted text
print(f"\n📝 Extracted Text ({len(extracted_text)} segments):")
print("-" * 50)
for i, text in enumerate(extracted_text, 1):
    print(f"{i:2d}. {text}")

print(f"\n✅ Tesseract completed successfully!")
